# 06 — Proximal, Distal & Departure

## Goal
Define the exact **price levels** of each zone and confirm that price departed from it with enough force to be meaningful.

---

## Proximal & Distal Levels
These define the two boundaries of a zone box.

| Zone Type | Proximal (near side — first touch on re-entry) | Distal (far side — back of zone) |
|-----------|------------------------------------------------|----------------------------------|
| **Demand** (RBR / DBR) | `base_high` — top of the base | `base_low` — bottom of the base |
| **Supply** (DBD / RBD) | `base_low` — bottom of the base | `base_high` — top of the base |

**Why this geometry?**  
- For a **demand zone**, price originally left *upward*. When it eventually returns, it approaches from above — so it hits the **top of the base first** (`base_high` = proximal).  
- For a **supply zone**, price originally left *downward*. When it returns, it approaches from below — hitting the **bottom of the base first** (`base_low` = proximal).

The proximal line is the level traders watch for re-entry; the distal line acts as a stop-loss reference.

---

## Departure Strength
After the base, price must leave with conviction. We measure the **peak excursion** (not closing price) over the first `N = 3` candles after the base:

$$\text{departure (demand)} = \max(\text{high}_{1 \dots N}) - \text{proximal}$$
$$\text{departure (supply)} = \text{proximal} - \min(\text{low}_{1 \dots N})$$

Then we require **two conditions** both to pass:

| Gate | Formula | Threshold |
|------|---------|-----------|
| ATR multiple | $\dfrac{\text{departure}}{\text{avg\_ATR}} \geq 1.5$ | Volatility-adjusted strength |
| Zone ratio | $\dfrac{\text{departure}}{\text{zone\_width}} \geq 1.0$ | Move large relative to the zone itself |

The zone-ratio gate catches cases where a move looks strong in ATR terms but barely clears the zone's own height.

---

## Scenario Comparison (A vs C)
- **Scenario A** — departure passes both gates cleanly → zone confirmed.
- **Scenario C** — departure ATR-ratio is below threshold → zone rejected ("weak departure").

---

## Visualization
The chart renders each confirmed zone as a colored rectangle:
- **Green box** = Demand zone
- **Red box** = Supply zone
- Horizontal lines mark `proximal` (solid) and `distal` (dashed) levels.  
- Labels show the formation type and departure ATR-ratio.


## 1. Constants — three groups
We split constants into three logical buckets so the dependency story is clear:

| Group | What it controls | Source |
|-------|------------------|--------|
| **Base** | what counts as a base candle / cluster | NB04 |
| **Leg** | how legs are measured and classified | NB05 |
| **Departure** | how strongly price must leave the base — *new in this notebook* |

The departure stage adds two new knobs: `DEPARTURE_CANDLES` (how far forward to look) and two strength gates (`DEPARTURE_ATR_MIN`, `DEPARTURE_RATIO_MIN`).


In [32]:
import pandas as pd
import numpy as np

# base-cluster (match NB04)
BASE_BODY_RATIO_MAX  = 0.50
BASE_MIN_CANDLES     = 1
BASE_MAX_CANDLES     = 5
BASE_MAX_ATR_WIDTH   = 2.5
BASE_COMPACTNESS_MAX = 0.80

# legs (match NB05)
LEG_CANDLES = 3
LEG_ATR_MIN = 1.5

# departure (NEW in this notebook)
DEPARTURE_CANDLES   = 3      # how many candles after the base to scan
DEPARTURE_ATR_MIN   = 1.5    # departure / ATR  must be at least this
DEPARTURE_RATIO_MIN = 1.0    # departure / zone_width must be at least this

df      = pd.read_csv("../fixtures_enhanced.csv", index_col=0, parse_dates=True)
labeled = pd.read_csv("../fixtures_labeled.csv",  index_col=0, parse_dates=True)

o_arr   = df["open"].values
h_arr   = df["high"].values
l_arr   = df["low"].values
c_arr   = df["close"].values
atr_arr = df["atr"].values

print(f"{len(df)} candles loaded")


98 candles loaded


## 2. Pipeline helpers — base clusters, legs, formation
Same logic as NB04 + NB05, restated here so the notebook is self-contained.
- `find_base_clusters()` / `cluster_ok()` come straight from NB04.
- `measure_legs()` returns `(lin_dir, lout_dir, avg_atr)` — slightly trimmed compared to NB05 since we only need the directions and the local volatility.
- `FORMATION_MAP` maps direction pairs to formation + zone-type.


In [33]:
FORMATION_MAP = {
    ("up",   "up"):   ("RBR", "demand"),
    ("down", "down"): ("DBD", "supply"),
    ("down", "up"):   ("DBR", "demand"),
    ("up",   "down"): ("RBD", "supply"),
}


def find_base_clusters():
    clusters = []
    i = 0
    while i < len(df):
        if not df["is_base"].iloc[i]:
            i += 1
            continue
        j = i
        while j + 1 < len(df) and df["is_base"].iloc[j + 1]:
            j += 1
        if BASE_MIN_CANDLES <= (j - i + 1) <= BASE_MAX_CANDLES:
            clusters.append((i, j))
        i = j + 1
    return clusters


def cluster_ok(bs, be):
    avg_atr = atr_arr[bs : be + 1].mean()
    width   = h_arr[bs : be + 1].max() - l_arr[bs : be + 1].min()
    compact = (c_arr[bs : be + 1].max() - c_arr[bs : be + 1].min()) / width if width > 0 else 1
    return (width <= BASE_MAX_ATR_WIDTH * avg_atr) and (compact <= BASE_COMPACTNESS_MAX)


def classify_move(net, avg_atr):
    t = LEG_ATR_MIN * avg_atr
    if net >=  t: return "up"
    if net <= -t: return "down"
    return "flat"


def measure_legs(bs, be):
    avg_atr = atr_arr[bs : be + 1].mean()

    if bs < LEG_CANDLES:
        return None
    lin_net = c_arr[bs - 1] - o_arr[bs - LEG_CANDLES]

    if be + LEG_CANDLES >= len(c_arr):
        return None
    lout_net = c_arr[be + LEG_CANDLES] - c_arr[be]

    return classify_move(lin_net, avg_atr), classify_move(lout_net, avg_atr), avg_atr


## 3. `proximal_distal` — the two boundary levels
Each zone has a **near side** (proximal — the line traders watch for re-entry) and a **far side** (distal — the stop-loss line).

For demand zones price comes *up* into them, so:
- proximal = `base_high` (you hit it first as price drops back)
- distal   = `base_low`  (back of the zone)

For supply zones the geometry flips.


In [34]:
def proximal_distal(bs, be, zone_type):
    bh = h_arr[bs : be + 1]
    bl = l_arr[bs : be + 1]
    if zone_type == "demand":
        return bh.max(), bl.min()    # proximal = top, distal = bottom
    else:                            # supply
        return bl.min(), bh.max()    # proximal = bottom, distal = top


## 4. `check_departure` — did price really leave the base?
A valid leg-out alone isn't enough. We also require price to **physically reach far enough** beyond the proximal level within the first `DEPARTURE_CANDLES` candles after the base.

We scan **peak excursion**, not just the closing price:
- Demand: maximum high in the window
- Supply: minimum low  in the window

Then evaluate two gates:

$$\text{dep\_atr}   = \frac{\text{departure}}{\overline{\text{ATR}}}  \geq 1.5$$
$$\text{dep\_ratio} = \frac{\text{departure}}{\text{zone\_width}}  \geq 1.0$$

Both must pass. The ATR gate ensures volatility-adjusted strength; the ratio gate ensures the move is large *relative to the size of the zone itself* (otherwise even a strong move into a huge base looks tiny).


In [35]:
def check_departure(proximal, be, zone_type, zone_width, avg_atr):
    end       = min(len(c_arr) - 1, be + DEPARTURE_CANDLES)
    candles_h = h_arr[be + 1 : end + 1]
    candles_l = l_arr[be + 1 : end + 1]
    if len(candles_h) == 0:
        return 0, 0, 0, False

    if zone_type == "demand":
        dep = candles_h.max() - proximal      # how far ABOVE the top of the base
    else:
        dep = proximal - candles_l.min()      # how far BELOW the bottom of the base

    dep_ratio = dep / zone_width if zone_width > 0 else 0
    dep_atr   = dep / avg_atr   if avg_atr   > 0 else 0
    passed    = (dep_ratio >= DEPARTURE_RATIO_MIN) and (dep_atr >= DEPARTURE_ATR_MIN)
    return round(dep, 3), round(dep_ratio, 2), round(dep_atr, 2), passed


## 5. Walkthrough — Scenario A (clean departure)
A textbook demand RBR. We expect the post-base candles to punch well above the proximal level. Both gates should pass comfortably.


In [36]:
rows_a    = labeled[labeled["scenario"] == "A_demand_RBR"]
base_a    = rows_a[rows_a["note"].str.startswith("BASE")]
bs_a      = df.index.get_loc(base_a.index[0])
be_a      = df.index.get_loc(base_a.index[-1])
avg_atr_a = atr_arr[bs_a : be_a + 1].mean()

prox_a, dist_a = proximal_distal(bs_a, be_a, "demand")
zw_a = abs(prox_a - dist_a)
dep_a, dep_ratio_a, dep_atr_a, passed_a = check_departure(prox_a, be_a, "demand", zw_a, avg_atr_a)

print("Scenario A — demand RBR")
print(f"  base highs   : {h_arr[bs_a : be_a + 1]}  → proximal = {prox_a}")
print(f"  base lows    : {l_arr[bs_a : be_a + 1]}  → distal   = {dist_a}")
print(f"  zone_width   : {zw_a:.3f}")
print(f"  avg ATR      : {avg_atr_a:.3f}")
print(f"  departure    : {dep_a:.3f}  (max high after base − proximal)")
print(f"  dep_ratio    : {dep_ratio_a}  (need >= {DEPARTURE_RATIO_MIN})")
print(f"  dep_atr      : {dep_atr_a}   (need >= {DEPARTURE_ATR_MIN})")
print(f"  passed       : {passed_a}")


Scenario A — demand RBR
  base highs   : [105.8 105.7 105.9]  → proximal = 105.9
  base lows    : [105.  104.9 105.1]  → distal   = 104.9
  zone_width   : 1.000
  avg ATR      : 1.587
  departure    : 5.600  (max high after base − proximal)
  dep_ratio    : 5.6  (need >= 1.0)
  dep_atr      : 3.53   (need >= 1.5)
  passed       : True


## 6. Walkthrough — Scenario C (weak departure → rejected)
Scenario C is deliberately the **counter-example**: the legs look fine in isolation, but the post-base candles barely poke above the proximal line. At least one gate should fail here.

This is exactly the case the departure filter is designed to catch.


In [37]:
rows_c    = labeled[labeled["scenario"] == "C_weak_dep"]
base_c    = rows_c[rows_c["note"].str.startswith("BASE")]
bs_c      = df.index.get_loc(base_c.index[0])
be_c      = df.index.get_loc(base_c.index[-1])
avg_atr_c = atr_arr[bs_c : be_c + 1].mean()

prox_c, dist_c = proximal_distal(bs_c, be_c, "demand")
zw_c = abs(prox_c - dist_c)
dep_c, dep_ratio_c, dep_atr_c, passed_c = check_departure(prox_c, be_c, "demand", zw_c, avg_atr_c)

print("Scenario C — weak departure")
print(f"  proximal     : {prox_c}  distal: {dist_c}  zone_width: {zw_c:.3f}")
print(f"  departure    : {dep_c:.3f}")
print(f"  dep_ratio    : {dep_ratio_c}  (need >= {DEPARTURE_RATIO_MIN})")
print(f"  dep_atr      : {dep_atr_c}   (need >= {DEPARTURE_ATR_MIN})")
print(f"  passed       : {passed_c}")


Scenario C — weak departure
  proximal     : 104.0  distal: 103.0  zone_width: 1.000
  departure    : 0.400
  dep_ratio    : 0.4  (need >= 1.0)
  dep_atr      : 0.24   (need >= 1.5)
  passed       : False


## 7. Full pipeline — every cluster end-to-end
Now we run the whole funnel:

```
find_base_clusters
        │
        ▼
   cluster_ok           ← tightness gates
        │
        ▼
   measure_legs         ← needs enough history + future
        │
        ▼
   FORMATION_MAP        ← legs must form one of 4 named formations
        │
        ▼
   proximal_distal      ← compute the zone box
        │
        ▼
   check_departure      ← strength gate
        │
        ▼
        zones[]
```

Every zone gets a `passed` flag so we can see what each gate rejected.


In [38]:
zones = []
for bs, be in find_base_clusters():
    if not cluster_ok(bs, be):
        continue
    result = measure_legs(bs, be)
    if result is None:
        continue
    lin_dir, lout_dir, avg_atr = result
    pair = FORMATION_MAP.get((lin_dir, lout_dir))
    if pair is None:
        continue
    formation, zone_type = pair

    prox, dist = proximal_distal(bs, be, zone_type)
    zw = abs(prox - dist)
    dep, dep_ratio, dep_atr, passed = check_departure(prox, be, zone_type, zw, avg_atr)

    zones.append(dict(
        scenario=labeled["scenario"].iloc[bs],
        formation=formation, zone_type=zone_type,
        bs=bs, be=be,
        proximal=prox, distal=dist, zone_width=zw,
        departure=dep, dep_ratio=dep_ratio, dep_atr=dep_atr,
        passed=passed,
    ))

passed_zones   = [z for z in zones if z["passed"]]
rejected_zones = [z for z in zones if not z["passed"]]

print(f"{len(passed_zones)} zones passed  |  {len(rejected_zones)} rejected by departure filter\n")
for z in zones:
    mark = "✓" if z["passed"] else "✗"
    print(f"  {mark} {z['zone_type']:6s} {z['formation']}  {z['scenario']:<20s}  "
          f"prox={z['proximal']:.2f}  dep_ratio={z['dep_ratio']}  dep_atr={z['dep_atr']}")


9 zones passed  |  0 rejected by departure filter

  ✓ demand RBR  A_demand_RBR          prox=105.90  dep_ratio=5.6  dep_atr=3.53
  ✓ supply RBD  A_demand_RBR          prox=109.00  dep_ratio=2.14  dep_atr=1.72
  ✓ supply DBD  B_supply_DBD          prox=105.80  dep_ratio=5.89  dep_atr=3.02
  ✓ demand RBR  D_wide_base           prox=115.80  dep_ratio=3.64  dep_atr=2.28
  ✓ demand RBR  E_doji_base           prox=120.90  dep_ratio=5.0  dep_atr=2.1
  ✓ demand RBR  E_doji_base           prox=125.40  dep_ratio=3.86  dep_atr=2.49
  ✓ demand DBR  H_DBR                 prox=135.70  dep_ratio=7.1  dep_atr=3.34
  ✓ demand RBR  H_DBR                 prox=140.10  dep_ratio=5.0  dep_atr=2.51
  ✓ supply RBD  I_RBD                 prox=144.60  dep_ratio=4.8  dep_atr=2.23


## 8. Visualization
Each zone is rendered as a rectangle with two horizontal levels extending into the departure window:
- **Solid dashed line** — proximal (near side, traders' entry trigger)
- **Dotted line** — distal (far side, stop-loss reference)
- Box fill **darker** for `passed`, **fainter** for rejected
- Each annotation shows the formation, status, and the two ratios so you can spot exactly which gate failed (if any).


In [39]:
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook_connected"

COLOR_BULL = "#26a69a"
COLOR_BEAR = "#ef5350"
BG         = "#131722"
GRID       = "#1e222d"
EDGE       = {"demand": "#26a69a", "supply": "#ef5350"}

fig = go.Figure()

fig.add_trace(go.Candlestick(
    x=df.index,
    open=df["open"], high=df["high"],
    low=df["low"],   close=df["close"],
    increasing_line_color=COLOR_BULL,
    decreasing_line_color=COLOR_BEAR,
    name="Price",
))

for z in zones:
    x0     = df.index[z["bs"]]
    x1     = df.index[z["be"]]
    c_edge = EDGE[z["zone_type"]]
    alpha  = "0.15" if z["passed"] else "0.06"
    fill   = f"rgba(38,166,154,{alpha})" if z["zone_type"] == "demand" else f"rgba(239,83,80,{alpha})"

    fig.add_vrect(x0=x0, x1=x1, fillcolor=fill,
                  line=dict(color=c_edge, width=1, dash="dot"))

    # extend the prox/distal lines into the departure window
    ext_idx = min(z["be"] + DEPARTURE_CANDLES + 1, len(df) - 1)
    x_ext   = df.index[ext_idx]

    fig.add_shape(type="line", x0=x0, x1=x_ext,
                  y0=z["proximal"], y1=z["proximal"],
                  line=dict(color=c_edge, width=1, dash="dash"))
    fig.add_shape(type="line", x0=x0, x1=x_ext,
                  y0=z["distal"], y1=z["distal"],
                  line=dict(color=c_edge, width=1, dash="dot"))

    status = "✓" if z["passed"] else "✗"
    label  = f"{z['formation']} {status}<br>ratio={z['dep_ratio']} atr={z['dep_atr']}"
    fig.add_annotation(
        x=x0, y=z["proximal"],
        text=label, showarrow=False,
        font=dict(size=8, color=c_edge),
        xanchor="left", yanchor="bottom",
    )

fig.update_layout(
    title=f"Proximal / Distal / Departure — {len(passed_zones)} passed, {len(rejected_zones)} rejected",
    height=580,
    plot_bgcolor=BG, paper_bgcolor=BG,
    font_color="white",
    xaxis_rangeslider_visible=False,
    hovermode="x unified",
    xaxis=dict(gridcolor=GRID, showgrid=True, zeroline=False),
    yaxis=dict(gridcolor=GRID, showgrid=True, zeroline=False, title="Price"),
)

fig.show()
